In [14]:
###RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [15]:
import sys
print(sys.executable)

c:\Users\ami05\OneDrive\Desktop\PolicyAgentX\PolicyAgentX\backend\venv\Scripts\python.exe


In [16]:
import langchain
import langchain_core
import langchain_community

print("LangChain:", langchain.__version__)
print("LangChain Core:", langchain_core.__version__)
print("LangChain Community:", langchain_community.__version__)

LangChain: 1.2.14
LangChain Core: 1.4.8
LangChain Community: 0.4.1


In [17]:
!pip show langchain
!pip show langchain-community
!pip show langchain-classic

Name: langchain
Version: 1.2.14
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\ami05\OneDrive\Desktop\PolicyAgentX\PolicyAgentX\backend\venv\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Name: langchain-community
Version: 0.4.1
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
License: MIT
Location: C:\Users\ami05\OneDrive\Desktop\PolicyAgentX\PolicyAgentX\backend\venv\Lib\site-packages
Requires: aiohttp, dataclasses-json, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, PyYAML, requests, SQLAlchemy, tenacity
Required-by: 
Name: langchain-classic
Version: 1.0.3
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\ami05\OneDrive\Desktop\PolicyAgentX\PolicyAgentX\backend

In [21]:
import os
from pathlib import Path

import langchain_core.document_loaders as core_document_loaders

if not hasattr(core_document_loaders, 'BaseLoader'):
    class BaseLoader:
        def load(self):
            return list(self.lazy_load())

        def lazy_load(self):
            raise NotImplementedError

    core_document_loaders.BaseLoader = BaseLoader

if not hasattr(core_document_loaders, 'BaseBlobParser'):
    class BaseBlobParser:
        pass

    core_document_loaders.BaseBlobParser = BaseBlobParser

# Backfill symbols expected by langchain-community on this kernel

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

In [22]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)


Found 7 PDF files to process

Processing: Cauvery-Water-Dispute.pdf
  ✓ Loaded 3 pages

Processing: Citizenship-Amendment-Act-2019.pdf
  ✓ Loaded 7 pages

Processing: Farmers (Empowerment and protection) bill, 2020.pdf
  ✓ Loaded 15 pages

Processing: India Farmer Demonstrations Continue Against Historic Agricultural Market Reforms_New Delhi_India_12-04-2020.pdf
  ✓ Loaded 4 pages

Processing: Language-dataset.pdf
  ✓ Loaded 15 pages

Processing: NewFarmActs2020.pdf
  ✓ Loaded 20 pages

Processing: The_Cauvery_River_Water_Dispute_A_Human_Rights_Per.pdf
  ✓ Loaded 7 pages

Total documents loaded: 71


In [23]:
all_pdf_documents

[Document(metadata={'producer': 'macOS Version 14.4.1 (Build 23E224) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20240406093916Z00'00'", 'title': 'Cauvery Water Dispute', 'author': 'Surbhi kataria', 'moddate': "D:20240406093916Z00'00'", 'source': '..\\data\\protests\\Cauvery-Water-Dispute.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Cauvery-Water-Dispute.pdf', 'file_type': 'pdf'}, page_content='Cauvery Water Dispute  The Kaveri is an interstate basin that originates in Karnataka and passes through Tamil Nadu and Pondicherry before draining into the Bay of Bengal. The total watershed of the Kaveri basin is 81,155 sq km, of which the river’s catchment area is about 34,273 sq km in Karnataka, 2,866 sq km in Kerala and the remaining 44,016 sq km in Tamil Nadu and Pondicherry. 1.The Harangi and Hemavati dams in Karnataka have been constructed on the Harangi and Hemavati rivers which are tributaries of the rivers Kaveri. The Krishna Raja Sagar Dam has b

In [24]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [25]:
chunks=split_documents(all_pdf_documents)
chunks

Split 71 documents into 396 chunks

Example chunk:
Content: Cauvery Water Dispute  The Kaveri is an interstate basin that originates in Karnataka and passes through Tamil Nadu and Pondicherry before draining into the Bay of Bengal. The total watershed of the K...
Metadata: {'producer': 'macOS Version 14.4.1 (Build 23E224) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20240406093916Z00'00'", 'title': 'Cauvery Water Dispute', 'author': 'Surbhi kataria', 'moddate': "D:20240406093916Z00'00'", 'source': '..\\data\\protests\\Cauvery-Water-Dispute.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Cauvery-Water-Dispute.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'macOS Version 14.4.1 (Build 23E224) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20240406093916Z00'00'", 'title': 'Cauvery Water Dispute', 'author': 'Surbhi kataria', 'moddate': "D:20240406093916Z00'00'", 'source': '..\\data\\protests\\Cauvery-Water-Dispute.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Cauvery-Water-Dispute.pdf', 'file_type': 'pdf'}, page_content='Cauvery Water Dispute  The Kaveri is an interstate basin that originates in Karnataka and passes through Tamil Nadu and Pondicherry before draining into the Bay of Bengal. The total watershed of the Kaveri basin is 81,155 sq km, of which the river’s catchment area is about 34,273 sq km in Karnataka, 2,866 sq km in Kerala and the remaining 44,016 sq km in Tamil Nadu and Pondicherry. 1.The Harangi and Hemavati dams in Karnataka have been constructed on the Harangi and Hemavati rivers which are tributaries of the rivers Kaveri. The Krishna Raja Sagar Dam has b

In [28]:
from langchain_community.vectorstores import Chroma
import numpy as np
from sklearn.feature_extraction.text import HashingVectorizer

class HashingEmbeddings:
    def __init__(self, n_features=384):
        self.vectorizer = HashingVectorizer(
            n_features=n_features,
            alternate_sign=False,
            norm=None
        )

    def embed_documents(self, texts):
        return self.vectorizer.transform(texts).toarray().astype(np.float32).tolist()

    def embed_query(self, text):
        return self.vectorizer.transform([text]).toarray().astype(np.float32)[0].tolist()

embedding_function = HashingEmbeddings()

vectorstore = Chroma(
    collection_name="rag_collection",
    embedding_function=embedding_function,
    persist_directory="./chroma_db"
)

C:\Users\ami05\AppData\Local\Temp\ipykernel_1824\387289115.py:21: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [30]:
import numpy as np
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import HashingVectorizer

In [31]:
import os
import warnings
import numpy as np
from typing import List
from sklearn.feature_extraction.text import HashingVectorizer

# Hide unnecessary warnings
warnings.filterwarnings("ignore")

class EmbeddingManager:
    """Handles document embedding generation using HashingVectorizer"""

    def __init__(self, n_features: int = 384):
        self.n_features = n_features
        self.model = HashingVectorizer(
            n_features=n_features,
            alternate_sign=False,
            norm=None
        )

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        print(f"Generating embeddings for {len(texts)} text(s)...")

        embeddings = self.model.transform(texts).toarray().astype(np.float32)

        print("✅ Embeddings generated successfully!")
        print(f"Shape: {embeddings.shape}")

        return embeddings


# Initialize the embedding manager
embedding_manager = EmbeddingManager()

In [32]:
chunks

[Document(metadata={'producer': 'macOS Version 14.4.1 (Build 23E224) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20240406093916Z00'00'", 'title': 'Cauvery Water Dispute', 'author': 'Surbhi kataria', 'moddate': "D:20240406093916Z00'00'", 'source': '..\\data\\protests\\Cauvery-Water-Dispute.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Cauvery-Water-Dispute.pdf', 'file_type': 'pdf'}, page_content='Cauvery Water Dispute  The Kaveri is an interstate basin that originates in Karnataka and passes through Tamil Nadu and Pondicherry before draining into the Bay of Bengal. The total watershed of the Kaveri basin is 81,155 sq km, of which the river’s catchment area is about 34,273 sq km in Karnataka, 2,866 sq km in Kerala and the remaining 44,016 sq km in Tamil Nadu and Pondicherry. 1.The Harangi and Hemavati dams in Karnataka have been constructed on the Harangi and Hemavati rivers which are tributaries of the rivers Kaveri. The Krishna Raja Sagar Dam has b

In [33]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

# (Optional) Generate embeddings just to inspect them
embeddings = embedding_manager.generate_embeddings(texts)

# Store documents in the vector database
vectorstore.add_documents(chunks)

Generating embeddings for 396 text(s)...
✅ Embeddings generated successfully!
Shape: (396, 384)


['fe07ba01-5293-47bc-b2e1-b78508c84234',
 '50f2b304-ba13-44cc-b16d-ab35c808eed6',
 '921a9666-6934-4aab-b619-5ec404f633eb',
 '96f8f020-3058-42ab-9965-69a4fb793dbd',
 '86c8913d-9396-4bfc-a985-d3eeeb780960',
 'e476aa29-fd4f-4f8f-bf8f-99e13d1d414a',
 '19d74df5-5b95-484b-a106-d29940247635',
 '2fd2b531-83c0-4055-889e-9d11560585b6',
 '1bd12074-8308-4b6e-8cc1-81088843738a',
 'a6596d1b-9e36-4a85-8389-5c982a924823',
 'c930e69b-a5ac-49b0-afeb-40d044429784',
 'b67f96be-34b1-4e1b-9f44-602f6631ba57',
 '39901fea-00b7-42b8-a29d-a59b35812b29',
 '096b92fc-e20e-44e5-9e7e-b2c393958dab',
 '68209f11-ad9f-4d32-be6b-429d390b378c',
 '031d16d2-3648-4501-a5ea-47729a0768ff',
 '007ab71c-4011-4349-b8f3-99abe8cc97c0',
 '6ea2fb25-9f2e-4751-81ef-7045ad9a6a67',
 '7311d266-406a-49b0-9c26-ddfdbc774812',
 '0edfcae3-f073-4b9c-a3ce-3ea495c21318',
 'b920fb6d-43cf-4a8d-91fa-aa6d2718d461',
 '7bdc20f2-9b99-4203-8f3b-23fccd897d92',
 '948e8858-5bc4-4339-92b9-ab5226b3bca5',
 '52b38546-1720-4c11-a226-2c4990acef5d',
 '788f0f63-a3ac-

In [34]:
###Retriver pipeline from vector store

In [35]:
from typing import List, Dict, Any
from langchain_community.vectorstores import Chroma
import numpy as np
from sklearn.feature_extraction.text import HashingVectorizer

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0):

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}")

        try:
            results = self.vector_store.similarity_search_with_score(
                query,
                k=top_k
            )

            retrieved_docs = []

            for i, (doc, score) in enumerate(results):

                retrieved_docs.append({
                    "id": i,
                    "content": doc.page_content,
                    "metadata": doc.metadata,
                    "similarity_score": score,
                    "rank": i + 1
                })

            print(f"Retrieved {len(retrieved_docs)} documents")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


if 'embedding_function' not in globals():
    class HashingEmbeddings:
        def __init__(self, n_features=384):
            self.vectorizer = HashingVectorizer(
                n_features=n_features,
                alternate_sign=False,
                norm=None
            )

        def embed_documents(self, texts):
            return self.vectorizer.transform(texts).toarray().astype(np.float32).tolist()

        def embed_query(self, text):
            return self.vectorizer.transform([text]).toarray().astype(np.float32)[0].tolist()

    embedding_function = HashingEmbeddings()

if 'vectorstore' not in globals():
    vectorstore = Chroma(
        collection_name="rag_collection",
        embedding_function=embedding_function,
        persist_directory="./chroma_db"
    )

if 'embedding_manager' not in globals():
    embedding_manager = EmbeddingManager()

# Create Retriever
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [36]:
rag_retriever

In [37]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5
Retrieved 5 documents


[{'id': 0,
  'content': 'against farmers shall be initiated against land of the farmer. In case the \nsponsor fails to pay the farmer, there is a provision for penalty extending to \none and a half times the amount owed. If a farmer reneges into the \nagreement, the recovery shall not exceed the actual cost incurred by the \nsponsor on account of any advance payment or cost of input supplied by \nhim. \nState governments have been given the power to make rules for carrying out \nprovisions of the Act, such as registration of a farming agreement. The Act \nkeeps scope to remove any diﬃculty in giving eﬀect to the provisions of this \nAct. \nEssential Commodities (Amendment) Act\nThe Essential Commodities Act has been modiﬁed for agriculture and food \nstuﬀ, including cereals, pulses, potato, onion, edible oilseeds and oils. The \nmodiﬁcation says that the Central government may regulate the supply of \nthe above commodities only under extraordinary circumstances, which may',
  'metadata

In [38]:
rag_retriever.retrieve("What are the various policies ")

Retrieving documents for query: 'What are the various policies '
Top K: 5
Retrieved 5 documents


[{'id': 0,
  'content': 'Citizenship Amendment Act 2019 (CAA)  \nThe Citizenship Amendment Bill was first introduced in 2016 by the Lok Sabha by amending the \nCitizenship Act of 1955. This bill was referred to a Joint Parliamentary Committee, whose report was later \nsubmitted on January 7, 2019. The Citizenship Amendment Bill was passed on January 8, 2019, by the Lok \nSabha which lapsed with the dissolution of the 16th Lok Sabha.   This Bill was introduced again on 9 \nDecember 2019 by the Minister of Home Affairs Amit Shah in the 17th Lok Sabha and was later passed on \n10 December 2019. The Rajya Sabha also passed the bill on 11th December.  \nThe CAA was passed to provide Indian citizenship to the illegal migrants who entered India on or before \n31st December 2014. The Act was passed for migrants of six different religions such a s Hindus, Sikhs, \nBuddhists, Jains, Parsis and Christians from Afghanistan, Bangladesh and Pakistan. Any individual will be',
  'metadata': {'creation